## Part B — fresh process

**Restart the runtime first** (Runtime → Restart session). The cells below read only files under
`/content`, so they run in a new Python process: that is the reload test the spec asks for, and it frees
the GPU (a 4B model twice does not fit a 15 GB T4).

In [ ]:
# ---- B1: reload the exported adapter in this fresh process -------------------------------------
import json, sys, os, math, gc, time, shutil, torch
PART_A = json.load(open("/content/part_a.json"))
B, EXP, MODE, SEED = PART_A["bundle_dir"], PART_A["export_dir"], PART_A["mode"], PART_A["seed"]
sys.path.insert(0, B)
assert torch.cuda.memory_allocated() == 0, "not a fresh process: restart the runtime first"
from unsloth import FastLanguageModel
from training import colab_helpers as H, prepare as P
m, proc = FastLanguageModel.from_pretrained(model_name=EXP, max_seq_length=P.MAX_TOKENS,
                                            load_in_4bit=False, load_in_16bit=True)
tok = getattr(proc, "tokenizer", proc)
FastLanguageModel.for_inference(m)
sample = json.load(open("/content/reload_sample.json"))
got = [H.generate_label(m, tok, x["messages"]) for x in sample]
RELOAD = {"reload_match": sum(g == x["expect"] for g, x in zip(got, sample)), "of": len(sample),
          "fingerprint_match": P.sha256(tok.chat_template) == PART_A["fingerprint"]["chat_template_sha256"],
          "weights_dtype": str(next(m.parameters()).dtype)}
print(RELOAD)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.10.5: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


`causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


{'reload_match': 5, 'of': 5, 'fingerprint_match': True, 'weights_dtype': 'torch.bfloat16'}


In [ ]:
# ---- B1s: candidate scores for calibration (spec §10): summed completion log-prob of every class ----
# val_calib fits the temperature, val_policy the threshold (both offline: python -m eval.calibration SCORES --write-policy);
# test / test_unseen / robustness are only reported. Dry runs and smoke score SCORE_CAP records per split.
import glob, yaml
SCORE_CAP = 20 if (MODE == "smoke" or PART_A.get("dry_run_steps")) else None
WANT = ("val_calib", "val_policy", "test", "test_unseen", "robustness")
classes = list(yaml.safe_load(open(f"{B}/config/taxonomy.yaml"))["classes"])
suf = H.candidate_suffixes(tok, classes)
prefix = PART_A["train_file"].split("train")[0]
recs, per = [], {}
for fn in sorted(glob.glob(f"{B}/datasets_ready/{prefix}*.jsonl")) + [f"{B}/datasets_ready/robustness_pilot.jsonl"]:
    if not os.path.exists(fn) or fn.endswith(PART_A["train_file"]):
        continue
    for line in open(fn):
        r = json.loads(line); sp = r["metadata"]["split"]
        if sp in WANT and (SCORE_CAP is None or per.get(sp, 0) < SCORE_CAP):
            per[sp] = per.get(sp, 0) + 1; recs.append(r)

@torch.no_grad()
def brute(msgs, i):  # reference without the cache trick: full forward pass of prompt + one class completion
    p = tok(tok.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False),
            add_special_tokens=False)["input_ids"]
    lp = torch.log_softmax(m(input_ids=torch.tensor([p + suf[i]], device="cuda")).logits[0].float(), -1)
    return sum(lp[len(p) - 1 + j, t].item() for j, t in enumerate(suf[i]))

out_path, t0, diffs, pdiffs = f"/content/SCORES_{MODE}.jsonl", time.time(), [], []
with open(out_path, "w") as fh:
    fh.write(json.dumps({"classes": classes, "mode": MODE, "fingerprint": PART_A["fingerprint"]}) + "\n")
    for n, r in enumerate(recs):
        s = H.class_logprobs(m, tok, r["messages"], suf)
        if n < 3:  # equivalence checks: unpruned replicated-cache scorer vs full forward pass on the true and top
            full = H.class_logprobs(m, tok, r["messages"], suf, margin=None)  # class; pruned vs unpruned probabilities
            for i in {classes.index(r["metadata"]["threat_type"]) if r["metadata"]["threat_type"] in classes else 0,
                      max(range(len(s)), key=s.__getitem__)}:
                diffs.append(abs(brute(r["messages"], i) - full[i]))
            pa, pb = torch.softmax(torch.tensor(s), 0), torch.softmax(torch.tensor(full), 0)
            pdiffs.append((pa - pb).abs().max().item())
        fh.write(json.dumps({"id": r["id"], "split": r["metadata"]["split"], "label": r["metadata"]["threat_type"],
                             "kind": r["metadata"].get("robustness_kind"), "scores": s}) + "\n")
SCORES = {"n": len(recs), "per_split": per, "s_per_record": round((time.time() - t0) / max(1, len(recs)), 3),
          "max_abs_diff_vs_bruteforce": max(diffs), "max_prob_diff_pruned": max(pdiffs), "file": out_path}
print(json.dumps(SCORES, indent=1))
assert SCORES["max_abs_diff_vs_bruteforce"] < 0.05, "cached scorer disagrees with the full forward pass"
assert SCORES["max_prob_diff_pruned"] < 1e-3, "pruned scorer changes the class probabilities"
del m; gc.collect(); torch.cuda.empty_cache()

{
 "n": 100,
 "per_split": {
  "test": 20,
  "test_unseen": 20,
  "val_policy": 20,
  "val_calib": 20,
  "robustness": 20
 },
 "s_per_record": 0.326,
 "max_abs_diff_vs_bruteforce": 0.00438321684487164,
 "max_prob_diff_pruned": 0.0047304630279541016,
 "file": "/content/SCORES_fast.jsonl"
}


AssertionError: pruned scorer changes the class probabilities

In [ ]:
# ---- B2: continuation test (smoke only): 20 steps straight vs 10 + save/reload + 10 ----------
RESUME = None
if MODE == "smoke" or PART_A.get("dry_run_steps"):
    from transformers import Trainer, TrainerCallback
    from datasets import Dataset

    class StopAt(TrainerCallback):
        def __init__(self, n): self.n = n
        def on_step_end(self, args, state, control, **kw):
            if state.global_step >= self.n: control.should_training_stop = True
            return control

    def run(out, stop=None, resume=None):
        m2, proc2 = FastLanguageModel.from_pretrained(model_name=PART_A["model_id"], revision=PART_A["revision"],
                                                      max_seq_length=P.MAX_TOKENS, load_in_4bit=False, load_in_16bit=True)
        tok2 = getattr(proc2, "tokenizer", proc2)
        m2 = FastLanguageModel.get_peft_model(m2, **H.LORA, random_state=SEED, max_seq_length=P.MAX_TOKENS)
        rows = H.encode(tok2, H.load_records(B, PART_A["train_file"])[:20 * 16])
        t = Trainer(model=m2, args=H.make_args(out, 20, 10, smoke=True, seed=SEED, evaluate=False),
                    train_dataset=Dataset.from_list(rows), data_collator=H.make_collate(tok2.pad_token_id),
                    callbacks=[StopAt(stop)] if stop else [])
        t.train(resume_from_checkpoint=resume)
        losses = {h["step"]: h["loss"] for h in t.state.log_history if "loss" in h}
        del t, m2; gc.collect(); torch.cuda.empty_cache()
        return losses
    full = run("/content/rt_full")
    run("/content/rt_half", stop=10)
    cont = run("/content/rt_half", resume="/content/rt_half/checkpoint-10")
    steps = [s for s in range(11, 21) if s in full and s in cont]
    diff = max(abs(full[s] - cont[s]) for s in steps)
    RESUME = {"steps_compared": steps, "uninterrupted": [full[s] for s in steps],
              "continued": [cont[s] for s in steps], "max_abs_loss_diff": diff,
              "tolerance": 0.05, "pass": bool(steps) and diff <= 0.05}
    print(json.dumps(RESUME, indent=1))

==((====))==  Unsloth 2026.10.5: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 320 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Unsloth: Will smartly offload gradients to save VRAM!


...

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 320 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Step,Training Loss
11,0.000026
12,0.000017
13,0.000015
14,0.000013
15,0.000012
16,0.000012
17,0.000011
18,0.000011
19,0.000011
20,0.000011


{
 "steps_compared": [
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20
 ],
 "uninterrupted": [
  2.613161996123381e-05,
  1.8587219528853893e-05,
  1.5041685401229188e-05,
  1.3110562576912344e-05,
  1.2622561371244956e-05,
  1.1996733519481495e-05,
  1.1803020242950879e-05,
  1.1575790267670527e-05,
  1.1634637303359341e-05,
  1.131279168475885e-05
 ],
 "continued": [
  2.5505047233309597e-05,
  1.748088834574446e-05,
  1.4895651474944316e-05,
  1.2629275261133444e-05,
  1.2288041943975259e-05,
  1.1896896467078477e-05,
  1.1329180779284798e-05,
  1.1177944543305784e-05,
  1.101403177017346e-05,
  1.1079598152718972e-05
 ],
 "max_abs_loss_diff": 1.1063311831094325e-06,
 "tolerance": 0.05,
 "pass": true
}


In [ ]:
# ---- B3: smoke report (the evidence for 'Notebook smoke-tested') -------------------------------
REPORT = {**PART_A, "reload": RELOAD, "resume_test": RESUME, "scores": SCORES}
REPORT["passed"] = bool(PART_A["train"]["global_steps"] >= 20
                        and all(math.isfinite(l) for _, l in PART_A["train"]["losses"])
                        and RELOAD["reload_match"] == RELOAD["of"] and RELOAD["fingerprint_match"]
                        and (RESUME is None or RESUME["pass"]))
name = ("gpu_smoke_test.json" if MODE == "smoke" else f"dryrun_{MODE}_report.json" if PART_A.get("dry_run_steps")
        else f"train_{MODE}_report.json")
json.dump(REPORT, open(f"/content/{name}", "w"), indent=1, default=str)
shutil.make_archive(f"/content/export_{MODE}", "zip", EXP)
if PART_A.get("drive_dir"):
    for f in (f"/content/{name}", SCORES["file"]):
        shutil.copy(f, PART_A["drive_dir"])
print("passed:", REPORT["passed"], "->", name, f"and {SCORES['file']} (download both into reports/; then locally:",
      f"python -m eval.calibration reports/{os.path.basename(SCORES['file'])} --write-policy)")

passed: True -> dryrun_fast_report.json and /content/SCORES_fast.jsonl (download both into reports/; then locally: python -m eval.calibration reports/SCORES_fast.jsonl --write-policy)
